# Building a simple Retrieval Generation System
This tutorial follows the tutorial from langchain to this topic ( https://python.langchain.com/docs/tutorials/rag/ ).
If you work in Windows go to System Settings and change to "developer mode".
(Just my personnel note: use the "conda python3.11.13" environment. )



In [ ]:
# Source - https://stackoverflow.com/a/52360659
# Posted by Davies Odu, modified by community. See post 'Timeline' for change history
# Retrieved 2026-03-11, License - CC BY-SA 4.0

from platform import python_version

print(python_version())


### LangSmith API key
Follow https://docs.smith.langchain.com/administration/how_to_guides/organization_management/create_account_api_key to create an account and to get an API key. 

You can set an environment variable.

Instead I created an .env file and stored there the LANGSMITH_API_KEY and the HUGGINGFACE_TOKEN.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os

LANGSMITH_API_KEY= os.getenv("LANGSMITH_API_KEY")
print("LANGSMITH_API_KEY: ", LANGSMITH_API_KEY)

### Install langchain and the integration of Huggingface

In [ ]:
#!pip install --upgrade langchain-text-splitters langchain-community langgraph
#!pip install langchain-huggingface
# !pip install langchain-classic
# !pip install --upgrade langchain
# !pip install --upgrade langsmith

# !pip install --upgrade langchain-huggingface

### Update the Jupyter Notebook
I had update the widgets of Jupyter Notebooks.

In [ ]:
#!pip install --upgrade ipywidgets

### Huggingface API
We need also a Huggingface API key, there are also named access token or Huggingface Secret. Follow https://huggingface.co/docs/hub/security-tokens to get one.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
HUGGINGFACEHUB_API_TOKEN = os.getenv("HUGGINGFACE_TOKEN")
print("HUGGINGFACEHUB_API_TOKEN: ", HUGGINGFACEHUB_API_TOKEN)

### Build a chatbot and use a inference provider from Huggingface
We use the Inference API from Huggingface and the model "deepseek-ai/DeepSeek-R1-0528:novita". DeepSeek-R1 is a reasoning model, therefore we can see the reasoning steps in the response.

In [ ]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=os.environ["HUGGINGFACE_TOKEN"],
)

completion1 = client.chat.completions.create(
    model="deepseek-ai/DeepSeek-R1-0528:novita",
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Reply with only the final answer."},
        {"role": "user",  "content": "What is the capital of France?" },
    ],
)

response = completion1.choices[0].message.content
print(response)

The version 3.2 of DeepSeek is also a reasoning model, but it is more focused on giving the final answer, therefore we get a more concise answer.

In [ ]:
completion2 = client.chat.completions.create(
    model="deepseek-ai/DeepSeek-V3.2:novita",
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Reply with only the final answer."},
        {"role": "user", "content": "What is the capital of France?"},
    ],
)

response = completion2.choices[0].message.content
print(response)

In [ ]:
# Comfortable single-message call
message = "What is the capital of France?"

def ask_chatbot(message: str, model: str = "deepseek-ai/DeepSeek-V3.2:novita") -> str:
    completion3 = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Reply with only the final answer."},
            {"role": "user", "content": message},
        ],
    )

    text = (completion3.choices[0].message.content or "").strip()

    # Remove optional reasoning block if present
    if "</think>" in text:
        text = text.split("</think>", 1)[1].strip()

    return text

answer = ask_chatbot(message)
print(answer)

In [ ]:
ask_chatbot("How is the weather today in Cologne?")

In [ ]:
ask_chatbot("Translate the following English text to German: 'The weather is nice today.'")

## Build a Retrieval Augemented Generation system

### Embeddings
In addition to the LLM we also need embeddings. They are used to convert all the texts (the user input and also the documents we provide) into numerical vectors. 

In [ ]:
#!pip install sentence-transformers

More informatin here: https://python.langchain.com/api_reference/huggingface/embeddings/langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings.html

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
import sentence_transformers

#model_name = "sentence-transformers/all-mpnet-base-v2"   #Alternatively, you can use "sentence-transformers/all-MiniLM-L6-v2", it is faster and smaller but less accurate
model_name = "sentence-transformers/all-MiniLM-L6-v2"   
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}
hf = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

The input is converted into a numerical vector with dimension 768.
https://python.langchain.com/api_reference/huggingface/embeddings/langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings.html

In [ ]:
print("embedded vector of the one sentence =", hf.embed_query("hello world"))
print("embedded vector of the input =", hf.embed_documents(["Hello world. Here I am. "]))
print("Dimension of the vectors =", len(hf.embed_query("hello world")))

### Vector store
Now we create a vector store. FAISS is a vector store offered by Langchain. 
https://docs.langchain.com/oss/python/integrations/vectorstores#faiss

In [ ]:
#!pip install faiss-cpu
#!pip install langchain-community

In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_dim = len(hf.embed_query("hello world"))
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
     embedding_function=hf,
     index=index,
     docstore=InMemoryDocstore(),
     index_to_docstore_id={},
 )

### Load documents with the relevant knowledge
We want to fill the vector store with the knowledge stored in some relevant document. 

Now we load the documents, an overview is here https://python.langchain.com/docs/integrations/document_loaders/ . 
We use PyPDF according to https://python.langchain.com/docs/integrations/document_loaders/pypdfloader/ . 

In [ ]:
#!pip install pypdf

I import here just the document "Time Schedule.pdf" from our course. 

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "data/BaTIN_Aushang_FAQs.pdf"
loader = PyPDFLoader(file_path)

In [ ]:
docs = loader.load()
docs[0]

Easier to read:

In [ ]:
import pprint

pprint.pp(docs[0].metadata)


To control the action we print the beginning of the first document:

In [ ]:
print(docs[0].page_content[:500])

#### Multiple documents
With the following code we can import several documents, see https://medium.com/@ohnonaoki95/rag-with-openai-and-langchain-using-multiple-pdf-documents-291a56f9d264

I import here additionally the documents "Learning Sources", "Meetings for the groups", "Organisation" and "Software".

In [ ]:
def create_file_paths(folder_path):
    dir_list = os.listdir(folder_path)
    full_dir_list = []
    for i in range(len(dir_list)):
        if dir_list[i].endswith(".pdf"):
            full_dir_list.append(folder_path + dir_list[i])
    return full_dir_list

def load_and_split_multiple_files(list_of_file_paths):
    output = []
    for i in range(len(list_of_file_paths)):
        loader = PyPDFLoader(list_of_file_paths[i])
        pages = loader.load_and_split()
        output.extend(pages)
    return output

# Load and process data.
pdf_file_path = "data/"  # CHANGE THIS LINE TO YOUR OWN FOLDER STRUCTURE
full_pdf_file_paths = create_file_paths(pdf_file_path)
print(full_pdf_file_paths)
docs = load_and_split_multiple_files(full_pdf_file_paths)

In [ ]:
for doc in docs:
    print(doc.page_content)

Now we split the document into chunks. We use the simple method to split after a given length. To avoid loosing information with the split we overlap each split. 

There are more sophisticated methods described in https://python.langchain.com/docs/concepts/text_splitters/ . 

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split document into {len(all_splits)} sub-documents.")

We convert all chunks into into embeddings, give them an own identifier and store them in the vector store. 
Usually the vector store is generated once and then stored. (see end of notebook for commands)

In [ ]:
document_ids = vector_store.add_documents(documents=all_splits)

print("First three identifier =", document_ids[:3])

### Retrieval and Generation
We take the prompt from a "hub". The prompt is stored here https://smith.langchain.com/hub/rlm/rag-prompt . This is the formulation of the prompt template:

"You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

Question: {question} 

Context: {context} 

Answer:"


In [ ]:
from langchain_core.prompts import PromptTemplate

template = (
        "You are an assistant for question-answering tasks. "
        "Use the following pieces of retrieved context to answer the question. "
        "If you don't know the answer, just say that you don't know. "
        "Use three sentences maximum and keep the answer concise.\n\n"
        "Question: {question}\n\n"
        "Context: {context}\n\n"
        "Answer:"
    )
prompt = PromptTemplate.from_template(template)

prompt_value = prompt.invoke(
    {"context": "(context goes here)", "question": "(question goes here)"}
 )

if hasattr(prompt_value, "to_messages"):
    messages = prompt_value.to_messages()
    if messages:
        print(messages[0].content)
else:
    if hasattr(prompt_value, "to_string"):
        print(prompt_value.to_string())
    else:
        print(str(prompt_value))

In [ ]:
from langchain_core.documents import Document
from typing_extensions import List, TypedDict


class State(TypedDict):
    question: str
    context: List[Document]
    answer: str

In "retrieve" we take our question and search in the vector store for similar content. This is the "context".

In "generate" we first collect all the context and then send it to the hub together with the question. It returns the answer to the question concerning the context. 

In [ ]:
def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {"context": retrieved_docs}


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    prompt_value = prompt.invoke({"question": state["question"], "context": docs_content})
    user_prompt = prompt_value.to_string() if hasattr(prompt_value, "to_string") else str(prompt_value)

    completion = client.chat.completions.create(
        model="deepseek-ai/DeepSeek-V3.2:novita",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Reply with only the final answer."},
            {"role": "user", "content": user_prompt},
        ],
    )

    answer = (completion.choices[0].message.content or "").strip()
    if "</think>" in answer:
        answer = answer.split("</think>", 1)[1].strip()

    return {"answer": answer}

We put both together in a "StateGraph". 

In [ ]:
from langgraph.graph import START, StateGraph

graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

We can visualize the control flow in a graph:

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

### Use the RAG:

The RAG gives back the final answer, but also in "context" the Document-id as the source of information. 

In [ ]:
result = graph.invoke({"question": "Who is responsible for the study program?"})

print(f'Context: {result["context"]}\n\n')
print(f'Answer: {result["answer"]}')

In [ ]:
result = graph.invoke({"question": "What is the difference between the programs 'Technische Informatik' and 'Informatik und Systems-Engineering'?"})

print(f'Answer: {result["answer"]}')

In [ ]:
result = graph.invoke({"question": "Can I be credited for past internships?"})

print(f'Answer: {result["answer"]}')

### Storing the vector_store
We can store the vector_store locally and load it the next time again. 

In [ ]:
vector_store.save_local("faiss_index")

We can load the vector_store named "faiss_index" from our local storage:

In [ ]:
# Load the FAISS index from the local storage
vector_store = FAISS.load_local(
    "faiss_index", hf, allow_dangerous_deserialization=True
)